# FedPer Server

> The core abstraction for `FedPer` server: https://arxiv.org/abs/1912.00818

In [ ]:
#| default_exp servers.fedper

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs
from fedai.optimizers.custom_optimizers import PerturbedGradientDescent

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_server("fedper")
class ServerFedPer(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
                 

### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerFedPer, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                if key.startswith("backbone"):
                    global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)

        current_global_state = self.model.state_dict()

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client_model = client_state.get('model', None)
            
            for key in client_model.keys():
                if key.startswith("backbone"):
                    client_model[key] = current_global_state[key]

            self.state_mgr.set_state(id, {
                    'model': client_model,
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()